# Pyomo Examples with ripopt

This notebook demonstrates using ripopt as a Pyomo solver via `SolverFactory('ripopt')`.
We compare results with IPOPT on several standard NLP problems.

In [1]:
import pyomo_ripopt
from pyomo.environ import *

## Example 1: Rosenbrock Function (Unconstrained)

Minimize $f(x,y) = 100(y - x^2)^2 + (1-x)^2$

Known solution: $(x^*, y^*) = (1, 1)$, $f^* = 0$

In [2]:
m = ConcreteModel()
m.x = Var(initialize=-1.2)
m.y = Var(initialize=1.0)
m.obj = Objective(expr=100*(m.y - m.x**2)**2 + (1 - m.x)**2)

# Solve with ripopt
solver = SolverFactory('ripopt')
result = solver.solve(m)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6e}")
print(f"  x = {value(m.x):.10f}, y = {value(m.y):.10f}")

# Solve with ipopt for comparison
m.x.set_value(-1.2)
m.y.set_value(1.0)
solver_ipopt = SolverFactory('ipopt')
result_ipopt = solver_ipopt.solve(m)
print(f"\nipopt:  {result_ipopt.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6e}")
print(f"  x = {value(m.x):.10f}, y = {value(m.y):.10f}")

ripopt: optimal
  obj = 5.395054e-21
  x = 1.0000000000, y = 1.0000000000



ipopt:  optimal
  obj = 3.743976e-21
  x = 0.9999999999, y = 0.9999999999


## Example 2: HS071 (Constrained NLP)

Minimize $x_1 x_4 (x_1 + x_2 + x_3) + x_3$

Subject to:
- $x_1 x_2 x_3 x_4 \geq 25$
- $x_1^2 + x_2^2 + x_3^2 + x_4^2 = 40$
- $1 \leq x_i \leq 5$

Known solution: $f^* \approx 17.0140$

In [3]:
def build_hs071():
    m = ConcreteModel()
    m.x1 = Var(bounds=(1, 5), initialize=1.0)
    m.x2 = Var(bounds=(1, 5), initialize=5.0)
    m.x3 = Var(bounds=(1, 5), initialize=5.0)
    m.x4 = Var(bounds=(1, 5), initialize=1.0)
    m.obj = Objective(expr=m.x1*m.x4*(m.x1 + m.x2 + m.x3) + m.x3)
    m.c1 = Constraint(expr=m.x1*m.x2*m.x3*m.x4 >= 25)
    m.c2 = Constraint(expr=m.x1**2 + m.x2**2 + m.x3**2 + m.x4**2 == 40)
    return m

# Solve with ripopt
m = build_hs071()
result = SolverFactory('ripopt').solve(m)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}")
print(f"  x = [{value(m.x1):.6f}, {value(m.x2):.6f}, {value(m.x3):.6f}, {value(m.x4):.6f}]")

# Solve with ipopt
m = build_hs071()
result = SolverFactory('ipopt').solve(m)
print(f"\nipopt:  {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}")
print(f"  x = [{value(m.x1):.6f}, {value(m.x2):.6f}, {value(m.x3):.6f}, {value(m.x4):.6f}]")

ripopt: optimal
  obj = 17.014017
  x = [1.000000, 4.743000, 3.821150, 1.379408]



ipopt:  optimal
  obj = 17.014017
  x = [1.000000, 4.743000, 3.821150, 1.379408]


## Example 3: Quadratic Program with Inequality Constraints

Minimize $(x - 1)^2 + (y - 2)^2$

Subject to: $x + y \leq 2$, $x, y \geq 0$

Known solution: $(x^*, y^*) = (0.5, 1.5)$, $f^* = 0.5$

In [4]:
def build_qp():
    m = ConcreteModel()
    m.x = Var(bounds=(0, None), initialize=0.5)
    m.y = Var(bounds=(0, None), initialize=0.5)
    m.obj = Objective(expr=(m.x - 1)**2 + (m.y - 2)**2)
    m.c1 = Constraint(expr=m.x + m.y <= 2)
    return m

m = build_qp()
result = SolverFactory('ripopt').solve(m)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}, x = {value(m.x):.6f}, y = {value(m.y):.6f}")

m = build_qp()
result = SolverFactory('ipopt').solve(m)
print(f"ipopt:  {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}, x = {value(m.x):.6f}, y = {value(m.y):.6f}")

ripopt: optimal
  obj = 0.500000, x = 0.500000, y = 1.500000


ipopt:  optimal
  obj = 0.500000, x = 0.500000, y = 1.500000


## Example 4: Linear Program

Maximize $3x + 5y$

Subject to: $x + 2y \leq 14$, $3x + 2y \leq 18$, $x, y \geq 0$

In [5]:
def build_lp():
    m = ConcreteModel()
    m.x = Var(bounds=(0, 10))
    m.y = Var(bounds=(0, 10))
    m.obj = Objective(expr=3*m.x + 5*m.y, sense=maximize)
    m.c1 = Constraint(expr=m.x + 2*m.y <= 14)
    m.c2 = Constraint(expr=3*m.x + 2*m.y <= 18)
    return m

m = build_lp()
result = SolverFactory('ripopt').solve(m)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}, x = {value(m.x):.6f}, y = {value(m.y):.6f}")

m = build_lp()
result = SolverFactory('ipopt').solve(m)
print(f"ipopt:  {result.solver.termination_condition}")
print(f"  obj = {value(m.obj):.6f}, x = {value(m.x):.6f}, y = {value(m.y):.6f}")

ripopt: optimal
  obj = 36.000000, x = 2.000000, y = 6.000000


ipopt:  optimal
  obj = 36.000000, x = 2.000000, y = 6.000000


## Example 5: Nonlinear Regression

Fit $y = a \cdot e^{b \cdot x}$ to noisy data using least squares.

In [6]:
import numpy as np

# Generate synthetic data
np.random.seed(42)
x_data = np.linspace(0, 2, 10)
y_data = 2.5 * np.exp(0.8 * x_data) + np.random.normal(0, 0.3, len(x_data))

def build_regression():
    m = ConcreteModel()
    m.a = Var(initialize=1.0)
    m.b = Var(initialize=0.5)
    m.obj = Objective(
        expr=sum((y_data[i] - m.a * exp(m.b * x_data[i]))**2 for i in range(len(x_data)))
    )
    return m

m = build_regression()
result = SolverFactory('ripopt').solve(m)
print(f"ripopt: {result.solver.termination_condition}")
print(f"  a = {value(m.a):.6f}, b = {value(m.b):.6f}  (true: a=2.5, b=0.8)")
print(f"  residual = {value(m.obj):.6f}")

m = build_regression()
result = SolverFactory('ipopt').solve(m)
print(f"\nipopt:  {result.solver.termination_condition}")
print(f"  a = {value(m.a):.6f}, b = {value(m.b):.6f}  (true: a=2.5, b=0.8)")
print(f"  residual = {value(m.obj):.6f}")

model.name="unknown";
    - termination condition: optimal
    - message from solver: ripopt 0.2.0\x3a Solved To Acceptable Level


ripopt: optimal
  a = 2.626218, b = 0.778529  (true: a=2.5, b=0.8)
  residual = 0.415375



ipopt:  optimal
  a = 2.626218, b = 0.778529  (true: a=2.5, b=0.8)
  residual = 0.415375
